# Arquivos texto: `fopen`, `fprintf`, `fscanf` e `fgets` — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

Até aqui os dados dos nossos programas nasciam no código ou no teclado e morriam com o programa. Este
tutorial retoma o `boletim.c` visto hoje — que lê as notas de `turma.txt` e grava `boletim.txt` — e
segue o caminho dos slides: abrir e fechar arquivos, o buffer que só chega ao disco no `fclose`, ler e
gravar com formato, ler uma linha de cada vez e ler um caractere de cada vez. Depois de cada bloco há
um exercício com teste automático; no fim vêm as tarefas de modificação e o desafio.

## Objetivos

Ao final deste tutorial você será capaz de:

- Abrir arquivos com `fopen` no modo certo (`"r"`, `"w"`, `"a"`), testar o `NULL` e explicar a falha com `perror`
- Explicar por que o que foi gravado só chega ao disco em `fflush`, `fclose` ou no fim normal do programa
- Ler e gravar dados com formato usando `fscanf` e `fprintf`, controlando o laço pelo valor devolvido
- Ler linha a linha com `fgets`, tratar o `'\n'` e as linhas maiores que o vetor, e separar campos com `sscanf`
- Ler caractere a caractere com `fgetc`, guardando o resultado num `int`


In [ ]:

# Confira se o gcc está disponível e defina a função que testa os exercícios
!gcc --version | head -1

import subprocess

def testa(comando, esperado, entrada=None, tempo=20):
    '''Roda o comando e compara a saída com o esperado.'''
    try:
        r = subprocess.run(comando, shell=True, capture_output=True, text=True,
                           input=entrada, timeout=tempo)
    except subprocess.TimeoutExpired:
        print(f'Demorou mais de {tempo} s: algum laço não termina?')
        return
    limpa = lambda t: '\n'.join(l.rstrip() for l in t.strip('\n').split('\n'))
    saida, esperado = limpa(r.stdout), limpa(esperado)
    if r.returncode != 0:
        print(f'O comando terminou com erro (status {r.returncode}; '
              '-11 ou 139 = segmentation fault).')
    if r.stderr:
        print('stderr:', r.stderr[:1500])
    if saida == esperado:
        print('OK!')
    else:
        print('Saída esperada:')
        print(esperado)
        print('Saída obtida:')
        print(saida)


## O programa da aula

Primeiro, o arquivo de entrada: matrícula e quatro notas por linha. Depois, o programa condutor
**exatamente como saiu da aula**. Antes de executar, escreva o que você espera ver na tela e o que
espera encontrar em `boletim.txt`.


In [ ]:
%%writefile turma.txt
202600101 8.0 6.5 9.0 7.5
202600102 5.0 7.0 4.0 7.0
202600103 9.0 10.0 8.5 9.5
202600104 3.5 4.0 6.0 2.5


In [ ]:
%%writefile boletim.c
/* boletim.c --- le as notas de turma.txt e escreve boletim.txt
   uso: ./boletim                         (turma.txt -> boletim.txt)
        ./boletim entrada.txt saida.txt */
#include <stdio.h>

#define NAV 4   /* avaliacoes por aluno */

int main(int argc, char *argv[])
{
    const char *nome_ent = argc > 1 ? argv[1] : "turma.txt";
    const char *nome_sai = argc > 2 ? argv[2] : "boletim.txt";

    FILE *ent = fopen(nome_ent, "r");
    if (ent == NULL) {
        perror(nome_ent);
        return 1;
    }
    FILE *sai = fopen(nome_sai, "w");
    if (sai == NULL) {
        perror(nome_sai);
        fclose(ent);
        return 1;
    }

    int mat, n = 0, aprovados = 0;
    double v[NAV];
    while (fscanf(ent, "%d %lf %lf %lf %lf",
                  &mat, &v[0], &v[1], &v[2], &v[3]) == 5) {
        double m = (v[0] + v[1] + v[2] + v[3]) / NAV;
        fprintf(sai, "%d %5.2f %s\n", mat, m,
                m >= 5.0 ? "aprovado" : "reprovado");
        n++;
        if (m >= 5.0) aprovados++;
    }
    fclose(ent);
    fclose(sai);

    printf("%d alunos lidos de %s, %d aprovados\n", n, nome_ent, aprovados);
    printf("boletim gravado em %s\n", nome_sai);
    return 0;
}


In [ ]:

!gcc -Wall -Wextra boletim.c -o boletim && ./boletim
!echo "--- boletim.txt:"; cat boletim.txt
!echo "--- com um nome errado:"; ./boletim turmas.txt


## 1. Abrir e fechar

`fopen(nome, modo)` devolve um `FILE *` — um ponteiro para a estrutura que guarda o buffer, a posição
atual e os indicadores de fim e de erro daquele arquivo. Todas as outras funções recebem esse ponteiro.

| modo | para quê | se o arquivo existe | se não existe |
|---|---|---|---|
| `"r"` | ler | lê desde o início | devolve `NULL` |
| `"w"` | gravar | **apaga o conteúdo** | cria vazio |
| `"a"` | acrescentar | grava sempre no fim | cria vazio |

Se `fopen` falha, devolve `NULL` e guarda o motivo em `errno`; `perror(nome)` imprime o nome e o motivo
na saída de erro (`stderr`). Usar o `NULL` como se fosse um arquivo dá *segmentation fault*.

O diário abaixo deveria acumular anotações. Execute-o duas vezes e veja o que sobra:


In [ ]:
%%writefile diario.c
/* diario.c --- acrescenta uma anotacao ao diario de estudos
   uso: ./diario "estudei fopen" */
#include <stdio.h>

int main(int argc, char *argv[])
{
    if (argc < 2) {
        fprintf(stderr, "uso: %s \"anotacao\"\n", argv[0]);
        return 1;
    }
    FILE *f = fopen("diario.txt", "w");
    if (f == NULL) {
        perror("diario.txt");
        return 1;
    }
    fprintf(f, "- %s\n", argv[1]);
    fclose(f);
    return 0;
}


In [ ]:

!gcc -Wall -Wextra diario.c -o diario
!./diario "estudei fopen"
!./diario "estudei fprintf"
!cat diario.txt


Só a última anotação: cada `fopen(..., "w")` apaga o arquivo. Troque `"w"` por `"a"` na célula de
`diario.c`, apague o diário com `!rm diario.txt` e rode de novo.

### O buffer

`fprintf` não grava no disco: copia os caracteres para um vetor dentro do `FILE`. A gravação de verdade
acontece quando o buffer enche, em `fflush(f)`, em `fclose(f)` ou no fim normal do programa (`return`
do `main` ou `exit`). Se o programa cair antes, o que estava no buffer se perde:


In [ ]:
%%writefile queda.c
/* queda.c --- o programa grava tres linhas e cai antes do fclose */
#include <stdio.h>
#include <signal.h>

int main(void)
{
    FILE *f = fopen("log.txt", "w");
    if (f == NULL) {
        perror("log.txt");
        return 1;
    }
    for (int i = 1; i <= 3; i++)
        fprintf(f, "passo %d concluido\n", i);
    fprintf(stderr, "3 passos registrados em log.txt\n");

    raise(SIGSEGV);    /* simula um segmentation fault: o fclose nunca acontece */

    fclose(f);
    return 0;
}


In [ ]:

!gcc -Wall -Wextra queda.c -o queda
!./queda
!wc -c log.txt


`log.txt` existe — o `fopen` o criou — mas tem zero bytes. A mensagem apareceu porque foi para `stderr`,
que não tem buffer. Experimente: coloque `fflush(f);` logo antes do `raise` e rode de novo.

### Exercício 1 — tabela de temperaturas

`./temperaturas ini fim passo saida` grava no arquivo `saida` uma linha para cada temperatura em graus
Celsius de `ini` até `fim`, de `passo` em `passo`, no formato `"%4d C = %5.1f F\n"`
($F = C \cdot 9/5 + 32$). Se o arquivo não puder ser criado, imprima o motivo com `perror` e termine com
status 1. No fim, feche o arquivo.


In [ ]:
%%writefile ex1.c
#include <stdio.h>
#include <stdlib.h>

/* Exercicio 1: ./temperaturas ini fim passo saida
   TODO: abra o arquivo de saida com "w" (teste o NULL), grave uma linha por
   temperatura com fprintf e feche o arquivo. */
int main(int argc, char *argv[])
{
    if (argc != 5) {
        fprintf(stderr, "uso: %s ini fim passo saida\n", argv[0]);
        return 1;
    }
    int ini = atoi(argv[1]), fim = atoi(argv[2]), passo = atoi(argv[3]);
    const char *nome = argv[4];

    /* TODO: implemente aqui */
    (void) ini;
    (void) fim;
    (void) passo;
    (void) nome;

    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra ex1.c -o ex1
print('de 0 a 100' + ':', end=' ')
testa('rm -f temp.txt; ./ex1 0 100 25 temp.txt && cat temp.txt', '   0 C =  32.0 F\n  25 C =  77.0 F\n  50 C = 122.0 F\n  75 C = 167.0 F\n 100 C = 212.0 F')
print('negativos' + ':', end=' ')
testa('rm -f t2.txt; ./ex1 -40 0 20 t2.txt && cat t2.txt', ' -40 C = -40.0 F\n -20 C =  -4.0 F\n   0 C =  32.0 F')
print('diretório que não existe' + ':', end=' ')
testa('./ex1 0 10 5 /nao/existe/t.txt; echo "status $?"', 'status 1')


## 2. Com formato: `fscanf` e `fprintf`

`fprintf(f, ...)` e `fscanf(f, ...)` são `printf` e `scanf` com um `FILE *` na frente. O `FILE` guarda
a **posição**: cada leitura continua de onde a anterior parou. Um espaço no formato do `fscanf` pula
qualquer quantidade de espaços, tabulações e `'\n'` — por isso o `fscanf` não percebe onde as linhas
terminam.

`fscanf` devolve quantos campos converteu, ou `EOF` se o arquivo acabou antes do primeiro. O laço deve
testar esse valor. Testar `feof` antes de ler parece natural, mas `feof` só fica verdadeiro **depois**
que uma leitura tenta passar do fim e falha. Veja o que acontece:


In [ ]:
%%writefile fim.c
/* fim.c --- o mesmo laco de boletim.c, controlado por feof */
#include <stdio.h>

int main(void)
{
    FILE *ent = fopen("turma.txt", "r");
    if (ent == NULL) {
        perror("turma.txt");
        return 1;
    }
    int mat, n = 0;
    double v[4];
    while (!feof(ent)) {
        fscanf(ent, "%d %lf %lf %lf %lf",
               &mat, &v[0], &v[1], &v[2], &v[3]);
        n++;
        printf("%d: %d\n", n, mat);
    }
    fclose(ent);
    return 0;
}


In [ ]:
!gcc -Wall -Wextra fim.c -o fim && ./fim


Quatro alunos, cinco voltas: na quinta, `fscanf` não leu nada e `mat` ficou com o valor anterior.
Depois do laço, porém, `feof` é útil: se o laço terminou e `feof(f)` é falso, a leitura parou em
algo que não era o esperado — e não no fim do arquivo.

### Exercício 2 — estatística de um arquivo de números

`./estat arquivo` lê números reais separados por espaços e quebras de linha, em quantidade
desconhecida, com `fscanf(f, "%lf", &x) == 1`. Imprima:

```
n = 6
soma = 33.50
media = 5.58
menor = -2.00
maior = 12.50
```

Se não houver nenhum número, imprima só `nenhum numero`. Se a leitura parar antes do fim do arquivo,
imprima as estatísticas do que foi lido e depois `parou depois de <n> numeros: ha algo que nao e numero`.


In [ ]:
%%writefile dados.txt
3.5 7 -2
12.5
   4.0     8.5


In [ ]:
%%writefile ruim.txt
1 2 3 x 5


In [ ]:
%%writefile vazio.txt



In [ ]:
%%writefile ex2.c
#include <stdio.h>

/* Exercicio 2: ./estat arquivo
   TODO: leia os numeros com fscanf(f, "%lf", &x) == 1, acumulando n, soma,
   menor e maior; depois do laco, use feof para saber por que ele parou. */
int main(int argc, char *argv[])
{
    if (argc != 2) {
        fprintf(stderr, "uso: %s arquivo\n", argv[0]);
        return 1;
    }
    FILE *f = fopen(argv[1], "r");
    if (f == NULL) {
        perror(argv[1]);
        return 1;
    }

    /* TODO: implemente aqui */

    fclose(f);
    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra ex2.c -o ex2
print('dados.txt' + ':', end=' ')
testa('./ex2 dados.txt', 'n = 6\nsoma = 33.50\nmedia = 5.58\nmenor = -2.00\nmaior = 12.50')
print('ruim.txt' + ':', end=' ')
testa('./ex2 ruim.txt', 'n = 3\nsoma = 6.00\nmedia = 2.00\nmenor = 1.00\nmaior = 3.00\nparou depois de 3 numeros: ha algo que nao e numero')
print('vazio.txt' + ':', end=' ')
testa('./ex2 vazio.txt', 'nenhum numero')
print('só negativos' + ':', end=' ')
testa('printf -- "-5 -1.5\\n-9" > neg.txt; ./ex2 neg.txt', 'n = 3\nsoma = -15.50\nmedia = -5.17\nmenor = -9.00\nmaior = -1.50')


## 3. Uma linha de cada vez: `fgets`

`fscanf` com `%s` para no primeiro espaço — e um campo faltando numa linha faz o `fscanf` continuar
lendo na linha seguinte, desalinhando o resto do arquivo. Para dados organizados em linhas, leia a
linha inteira com `fgets` e só depois separe os campos.

`fgets(s, n, f)` lê até um `'\n'` (que **fica** em `s`) ou até completar `n - 1` caracteres, sempre
termina com `'\0'` e devolve `NULL` no fim do arquivo. Uma linha maior que o vetor chega em pedaços.
Veja com um vetor de 24 posições:


In [ ]:
%%writefile turma.csv
202600101;Ana Souza;8.0;6.5;9.0;7.5
202600102;Bruno Lima;5.0;7.0;4.0;7.0
202600103;Carla Menezes Santos;9.0;10.0;8.5;9.5
202600104;Davi Rocha;3.5;4.0;6.0;2.5


In [ ]:
%%writefile linhas.c
/* linhas.c --- le turma.csv linha a linha, com um buffer pequeno */
#include <stdio.h>

int main(void)
{
    FILE *ent = fopen("turma.csv", "r");
    if (ent == NULL) {
        perror("turma.csv");
        return 1;
    }
    char linha[24];
    while (fgets(linha, sizeof linha, ent) != NULL)
        printf("[%s]\n", linha);
    fclose(ent);
    return 0;
}


In [ ]:
!gcc -Wall -Wextra linhas.c -o linhas && ./linhas


O `]` aparece na linha de baixo quando o `'\n'` do arquivo veio dentro do vetor. Um pedaço **sem**
`'\n'` no fim quer dizer: a linha ainda não acabou (ou é a última linha, sem `'\n'`). Para tirar o
`'\n'`, `linha[strcspn(linha, "\n")] = '\0';`.

### Exercício 3 — numerar as linhas

`./numera entrada saida` copia `entrada` para `saida` pondo na frente de cada linha o seu número, no
formato `"%3d: "`. O vetor do `fgets` tem **16** posições de propósito: uma linha comprida chega em
vários pedaços, mas deve receber um número só — numere um pedaço apenas se o pedaço anterior terminou
em `'\n'` (ou se for o primeiro). Use `fputs(pedaco, saida)` para copiar o pedaço sem alterá-lo.


In [ ]:
%%writefile texto.txt
Arquivo texto
e uma fileira de caracteres dividida em linhas

cada linha termina com um caractere especial
fim


In [ ]:
%%writefile ex3.c
#include <stdio.h>
#include <string.h>

#define TAM 16   /* pequeno de proposito: linhas compridas chegam em pedacos */

/* Exercicio 3: ./numera entrada saida
   TODO: leia com fgets(pedaco, TAM, ent); escreva "%3d: " so no comeco de
   cada linha e copie o pedaco com fputs. */
int main(int argc, char *argv[])
{
    if (argc != 3) {
        fprintf(stderr, "uso: %s entrada saida\n", argv[0]);
        return 1;
    }
    FILE *ent = fopen(argv[1], "r");
    if (ent == NULL) {
        perror(argv[1]);
        return 1;
    }
    FILE *sai = fopen(argv[2], "w");
    if (sai == NULL) {
        perror(argv[2]);
        fclose(ent);
        return 1;
    }
    char pedaco[TAM];

    /* TODO: implemente aqui */
    (void) pedaco;

    fclose(ent);
    fclose(sai);
    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra ex3.c -o ex3
print('texto.txt' + ':', end=' ')
testa('rm -f num.txt; ./ex3 texto.txt num.txt && cat num.txt', '  1: Arquivo texto\n  2: e uma fileira de caracteres dividida em linhas\n  3:\n  4: cada linha termina com um caractere especial\n  5: fim')
print('sem \\n no fim' + ':', end=' ')
testa('printf "a\\nb" > semfim.txt; rm -f num2.txt; ./ex3 semfim.txt num2.txt && cat num2.txt', '  1: a\n  2: b')
print('exatamente 15 caracteres' + ':', end=' ')
testa('printf "123456789012345\\nx\\n" > q.txt; rm -f num3.txt; ./ex3 q.txt num3.txt && cat num3.txt', '  1: 123456789012345\n  2: x')


### `fgets` + `sscanf`

Com a linha inteira na mão, `sscanf` — o `fscanf` de uma *string* — separa os campos. Se a linha
estiver mal formada, só ela é descartada. `%49[^;]` lê até 49 caracteres que não sejam `;`, espaços
incluídos:


In [ ]:
%%writefile boletim_csv.c
/* boletim_csv.c --- le turma.csv (com nomes) e escreve boletim.csv
   uso: ./boletim_csv                         (turma.csv -> boletim.csv)
        ./boletim_csv entrada.csv saida.csv */
#include <stdio.h>
#include <string.h>

#define NAV 4

int main(int argc, char *argv[])
{
    const char *nome_ent = argc > 1 ? argv[1] : "turma.csv";
    const char *nome_sai = argc > 2 ? argv[2] : "boletim.csv";

    FILE *ent = fopen(nome_ent, "r");
    if (ent == NULL) {
        perror(nome_ent);
        return 1;
    }
    FILE *sai = fopen(nome_sai, "w");
    if (sai == NULL) {
        perror(nome_sai);
        fclose(ent);
        return 1;
    }

    char linha[256], nome[50];
    int mat, nlinha = 0, n = 0;
    double v[NAV];
    while (fgets(linha, sizeof linha, ent) != NULL) {
        nlinha++;
        linha[strcspn(linha, "\n")] = '\0';      /* tira o '\n' */
        if (sscanf(linha, "%d;%49[^;];%lf;%lf;%lf;%lf", &mat, nome,
                   &v[0], &v[1], &v[2], &v[3]) != 6) {
            fprintf(stderr, "%s:%d: linha ignorada: \"%s\"\n",
                    nome_ent, nlinha, linha);
            continue;
        }
        double m = (v[0] + v[1] + v[2] + v[3]) / NAV;
        fprintf(sai, "%d;%s;%.2f;%s\n", mat, nome, m,
                m >= 5.0 ? "aprovado" : "reprovado");
        n++;
    }
    fclose(ent);
    fclose(sai);

    printf("%d alunos gravados em %s\n", n, nome_sai);
    return 0;
}


In [ ]:

!gcc -Wall -Wextra boletim_csv.c -o boletim_csv && ./boletim_csv && cat boletim.csv


### Exercício 4 — filtrar um CSV

`./aprovados entrada.csv saida.csv` copia para a saída, **sem alterar**, só as linhas dos alunos com
média das quatro notas $\geq 5$, na ordem da entrada. Uma linha que não tenha os seis campos
(matrícula, nome e quatro notas) vai para `stderr` como `entrada.csv:<número da linha>: linha mal formada`
e não conta como aluno. No fim, imprima `<a> de <n> alunos aprovados`.


In [ ]:
%%writefile turma2.csv
202600101;Ana Souza;8.0;6.5;9.0;7.5
202600102;Bruno Lima;5.0;7.0;4.0;7.0
202600105;Elisa Prado;4.0;sete;5.0;6.0
202600103;Carla Menezes Santos;9.0;10.0;8.5;9.5
202600104;Davi Rocha;3.5;4.0;6.0;2.5
202600106;Fabio Reis;9.0;9.0


In [ ]:
%%writefile ex4.c
#include <stdio.h>
#include <string.h>

/* Exercicio 4: ./aprovados entrada.csv saida.csv
   TODO: leia cada linha com fgets, separe os campos com sscanf
   ("%d;%49[^;];%lf;%lf;%lf;%lf"), e copie a linha original para a saida se
   a media for >= 5. Linhas mal formadas: mensagem em stderr. */
int main(int argc, char *argv[])
{
    if (argc != 3) {
        fprintf(stderr, "uso: %s entrada.csv saida.csv\n", argv[0]);
        return 1;
    }
    FILE *ent = fopen(argv[1], "r");
    if (ent == NULL) {
        perror(argv[1]);
        return 1;
    }
    FILE *sai = fopen(argv[2], "w");
    if (sai == NULL) {
        perror(argv[2]);
        fclose(ent);
        return 1;
    }

    /* TODO: implemente aqui */

    fclose(ent);
    fclose(sai);
    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra ex4.c -o ex4
print('turma2.csv' + ':', end=' ')
testa('rm -f apr.csv; ./ex4 turma2.csv apr.csv 2>/dev/null && cat apr.csv', '3 de 4 alunos aprovados\n202600101;Ana Souza;8.0;6.5;9.0;7.5\n202600102;Bruno Lima;5.0;7.0;4.0;7.0\n202600103;Carla Menezes Santos;9.0;10.0;8.5;9.5')
print('mensagens de erro' + ':', end=' ')
testa('./ex4 turma2.csv apr.csv 2>&1 >/dev/null', 'turma2.csv:3: linha mal formada\nturma2.csv:6: linha mal formada')
print('média exatamente 5' + ':', end=' ')
testa('printf "1;Gil;5;5;5;5\\n2;Ivo;4.99;5;5;5\\n" > m5.csv; ./ex4 m5.csv apr5.csv && cat apr5.csv', '1 de 2 alunos aprovados\n1;Gil;5;5;5;5')


### Exercício 5 — arquivo de configuração

Programas costumam ler opções de um arquivo `chave = valor`. `./config arquivo chave` imprime o valor
da chave entre aspas — `"800"` — ou, se a chave não existir, escreve `<chave>: chave nao encontrada`
em `stderr` e termina com status 1. Regras:

- linhas em branco e linhas cujo primeiro caractere não-espaço é `#` são ignoradas;
- espaços antes da chave, em volta do `=` e no fim do valor não fazem parte de nada;
- o valor pode ter espaços no meio (`Batalha Naval`).

Implemente `le_par`, que recebe uma linha (já sem o `'\n'`) e devolve 1 se ela tem um par chave/valor.
Uma dica para o `sscanf`: `" %63[^= ] = %127[^\n]"`.


In [ ]:
%%writefile jogo.cfg
# configuracao do jogo
# largura = 1024
largura = 800
altura=600

titulo = Batalha Naval   
  # comentario recuado
   dificuldade =   dificil


In [ ]:
%%writefile ex5.c
#include <stdio.h>
#include <string.h>

/* Exercicio 5: ./config arquivo chave
   TODO: implemente le_par. Devolve 1 se a linha tem "chave = valor"
   (copiando-os para chave e valor, sem espacos nas pontas) e 0 se a linha
   esta em branco, e um comentario ou nao tem '='. */
int le_par(const char *linha, char chave[64], char valor[128])
{
    /* TODO: implemente aqui */
    (void) linha;
    (void) chave;
    (void) valor;
    return 0;
}

int main(int argc, char *argv[])
{
    if (argc != 3) {
        fprintf(stderr, "uso: %s arquivo chave\n", argv[0]);
        return 1;
    }
    FILE *f = fopen(argv[1], "r");
    if (f == NULL) {
        perror(argv[1]);
        return 1;
    }
    char linha[256], chave[64], valor[128];
    while (fgets(linha, sizeof linha, f) != NULL) {
        linha[strcspn(linha, "\n")] = '\0';
        if (le_par(linha, chave, valor) && strcmp(chave, argv[2]) == 0) {
            printf("\"%s\"\n", valor);
            fclose(f);
            return 0;
        }
    }
    fclose(f);
    fprintf(stderr, "%s: chave nao encontrada\n", argv[2]);
    return 1;
}


In [ ]:
!gcc -g -Wall -Wextra ex5.c -o ex5
print('largura' + ':', end=' ')
testa('./ex5 jogo.cfg largura', '"800"')
print('altura' + ':', end=' ')
testa('./ex5 jogo.cfg altura', '"600"')
print('titulo' + ':', end=' ')
testa('./ex5 jogo.cfg titulo', '"Batalha Naval"')
print('dificuldade' + ':', end=' ')
testa('./ex5 jogo.cfg dificuldade', '"dificil"')
print('cor' + ':', end=' ')
testa('./ex5 jogo.cfg cor 2>/dev/null; echo "status $?"', 'status 1')


## 4. Um caractere de cada vez: `fgetc`

`fgetc(f)` devolve o próximo caractere do arquivo, e `fputc(c, f)` grava um. O resultado de `fgetc`
vai num **`int`**: um arquivo pode conter qualquer um dos 256 valores de um byte, e `EOF` precisa ser
diferente de todos eles. Com `char c`, o laço ou para no meio do arquivo ou nunca termina, conforme o
`char` da máquina tenha sinal ou não.

O `conta.c` da aula conta linhas, palavras e bytes, como o `wc`:


In [ ]:
%%writefile conta.c
/* conta.c --- conta linhas, palavras e bytes de um arquivo, como o wc
   uso: ./conta arquivo */
#include <stdio.h>
#include <ctype.h>

int main(int argc, char *argv[])
{
    if (argc != 2) {
        fprintf(stderr, "uso: %s arquivo\n", argv[0]);
        return 1;
    }
    FILE *f = fopen(argv[1], "r");
    if (f == NULL) {
        perror(argv[1]);
        return 1;
    }
    int c;                       /* int, nao char: EOF nao e um caractere */
    long linhas = 0, palavras = 0, bytes = 0;
    int dentro = 0;
    while ((c = fgetc(f)) != EOF) {
        bytes++;
        if (c == '\n')
            linhas++;
        if (isspace(c))
            dentro = 0;
        else if (!dentro) {
            dentro = 1;
            palavras++;
        }
    }
    fclose(f);
    printf("%ld %ld %ld %s\n", linhas, palavras, bytes, argv[1]);
    return 0;
}


In [ ]:
!gcc -Wall -Wextra conta.c -o conta && ./conta turma.csv && wc turma.csv


### Exercício 6 — frequência das letras

`./letras arquivo` conta quantas vezes cada letra de `a` a `z` aparece, sem distinguir maiúsculas de
minúsculas, e imprime uma linha `<letra>: <contagem>` para cada letra que aparece pelo menos uma vez, em
ordem alfabética, seguida de `total: <n> letras`. Letras acentuadas ocupam dois bytes em UTF-8 e
nenhum deles está entre `'a'` e `'z'`: elas ficam de fora.


In [ ]:
%%writefile fruta.txt
Abacaxi e Banana!


In [ ]:
%%writefile acento.txt
Ação é útil


In [ ]:
%%writefile ex6.c
#include <stdio.h>
#include <ctype.h>

/* Exercicio 6: ./letras arquivo
   TODO: leia com fgetc (em um int!), conte cada letra com tolower e um
   vetor de 26 contadores, e imprima as que aparecem. */
int main(int argc, char *argv[])
{
    if (argc != 2) {
        fprintf(stderr, "uso: %s arquivo\n", argv[0]);
        return 1;
    }
    FILE *f = fopen(argv[1], "r");
    if (f == NULL) {
        perror(argv[1]);
        return 1;
    }

    /* TODO: implemente aqui */

    fclose(f);
    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra ex6.c -o ex6
print('fruta.txt' + ':', end=' ')
testa('./ex6 fruta.txt', 'a: 6\nb: 2\nc: 1\ne: 1\ni: 1\nn: 2\nx: 1\ntotal: 14 letras')
print('acento.txt' + ':', end=' ')
testa('./ex6 acento.txt', 'a: 1\ni: 1\nl: 1\no: 1\nt: 1\ntotal: 5 letras')
print('turma.csv' + ':', end=' ')
testa('./ex6 turma.csv', 'a: 9\nb: 1\nc: 2\nd: 1\ne: 3\nh: 1\ni: 2\nl: 2\nm: 2\nn: 4\no: 4\nr: 3\ns: 4\nt: 1\nu: 2\nv: 1\nz: 2\ntotal: 44 letras')


## Tarefas: altere o programa da aula

Volte à célula de `boletim.c`, faça cada alteração, reexecute e confira se a saída bate com o que você
previu.

1. **O boletim na tela.** Se o nome da saída for `"-"`, grave em `stdout`: `./boletim turma.txt -`.
   Basta escolher o `FILE *` certo — mas `stdout` não foi aberto por você: ele deve ser fechado no fim?
2. **Reprovados à parte.** Abra um terceiro arquivo, `reprovados.txt`, e grave nele só a matrícula de
   quem ficou abaixo de 5. Se essa abertura falhar, quem precisa ser fechado antes de sair?
3. **A média da turma, com `"a"`.** No fim do boletim, grave a linha `media da turma: 6.69`. Depois troque
   o modo da saída para `"a"` e rode o programa duas vezes. O que ficou em `boletim.txt`?

## Desafio: intercalar duas listas de presença

Dois arquivos trazem as matrículas presentes em duas aulas, uma por linha, em ordem crescente e sem
repetição dentro de cada arquivo. `./intercala a.txt b.txt todos.txt` grava em `todos.txt` todas as
matrículas, em ordem crescente e sem repetição, e imprime

```
so em a: <x>, so em b: <y>, nas duas: <z>
```

A regra do jogo: guarde na memória **uma** matrícula de cada arquivo por vez — nada de vetores. Compare
as duas, grave a menor e leia a próxima só do arquivo de onde ela veio (se forem iguais, grave uma vez
e avance nos dois). Quando um arquivo acabar, copie o resto do outro.


In [ ]:
%%writefile a.txt
101
103
104


In [ ]:
%%writefile b.txt
102
103
105


In [ ]:
%%writefile intercala.c
#include <stdio.h>

/* Desafio: ./intercala a.txt b.txt todos.txt
   TODO: intercale os dois arquivos guardando so uma matricula de cada. */
int main(int argc, char *argv[])
{
    if (argc != 4) {
        fprintf(stderr, "uso: %s a.txt b.txt saida.txt\n", argv[0]);
        return 1;
    }

    /* TODO: sua solucao aqui */

    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra intercala.c -o intercala
import random
random.seed(512)
A = sorted(random.sample(range(202600000, 202620000), 3000))
B = sorted(random.sample(range(202600000, 202620000), 5000))
open('grandeA.txt', 'w').write(''.join(f'{x}\n' for x in A))
open('grandeB.txt', 'w').write(''.join(f'{x}\n' for x in B))
sa, sb = set(A), set(B)
todos = sorted(sa | sb)
open('vazio2.txt', 'w').close()
casos = {
    'o dos slides': ('./intercala a.txt b.txt todos.txt && cat todos.txt',
        'so em a: 2, so em b: 2, nas duas: 1\n101\n102\n103\n104\n105'),
    'um vazio': ('./intercala vazio2.txt b.txt t2.txt && cat t2.txt',
        'so em a: 0, so em b: 3, nas duas: 0\n102\n103\n105'),
    'iguais': ('./intercala a.txt a.txt t3.txt && cat t3.txt',
        'so em a: 0, so em b: 0, nas duas: 3\n101\n103\n104'),
    'grandes': ('./intercala grandeA.txt grandeB.txt t4.txt && cat t4.txt',
        f'so em a: {len(sa - sb)}, so em b: {len(sb - sa)}, nas duas: {len(sa & sb)}\n'
        + '\n'.join(map(str, todos))),
}
for nome, (cmd, esperado) in casos.items():
    print(nome + ':', end=' ')
    r = testa(cmd, esperado)


Depois de passar no teste, pense: quanto de memória o seu programa usa para intercalar dois arquivos de
um milhão de linhas cada? É assim que se ordena um arquivo maior que a memória do computador: ordenam-se
pedaços que cabem na memória, gravam-se os pedaços ordenados em arquivos e intercalam-se os arquivos.

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. Os pontos principais desta aula estão em:

- Kernighan e Ritchie (1988), *The C Programming Language*, §1.5 — entrada e saída de caracteres e o
  `EOF` num `int`; §7.5 — acesso a arquivos; §7.6 — `stderr` e `exit`; §7.7 — `fgets` e `fputs`;
- King (2008), *C Programming: A Modern Approach*, cap. 22 — fluxos (§22.1), operações com arquivos
  (§22.2), E/S formatada (§22.3), de caracteres (§22.4), de linhas (§22.5) e em *strings* (§22.8);
- Deitel e Deitel (2011), *C: como programar*, cap. 11 — processamento de arquivos;
- ISO/IEC 9899:2018, §7.21 — `<stdio.h>`: `fopen` (§7.21.5.3), `fscanf` (§7.21.6.2), `fgets`
  (§7.21.7.2), `feof` (§7.21.10.2), `perror` (§7.21.10.4);
- Kerrisk (2010), *The Linux Programming Interface*, §13.2 — o buffer da `stdio`;
- Kernighan e Pike (1999), *The Practice of Programming*, §4.1 — lendo arquivos CSV;
- Knuth (1998), *The Art of Computer Programming*, vol. 3, §5.2.4 — intercalação (Algoritmo M).
